# PrimeGate -- BoundaryAlarm: the Crossing-Detector Primitive

**ValaQuenta Engine -- BoundaryAlarm, PrimeGateEngine**
**Date:** 2026-07-12
**Status:** REVISED -- the alarm is the point, not a test of any dataset.

---

### The Primitive

BoundaryAlarm fires once per crossing of a boundary condition, blind to
everything else -- magnitude, spacing, direction of approach. pi(x) (the
prime gate) and the Holcus FIRING signal (Ainulindale wiki/44, a computation
reaching sigma=1/2) are the SAME primitive, run on different streams. This
notebook builds the general alarm first, then shows the prime instantiation
and the gap channel it deliberately leaves out.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../.."))
import math
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 120, "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False})

from prime_gate import PrimeGateEngine, BoundaryAlarm, sigma_half_alarm

print("BoundaryAlarm and PrimeGateEngine loaded.")

## Section 1 -- The General Primitive

Not prime-specific. Demonstrated here on a sigma-trajectory crossing 1/2 --
the Holcus-style firing condition -- using the identical class that will
also drive the prime gate below.

In [ ]:
traj = [0.1, 0.3, 0.45, 0.4999995, 0.5000003, 0.7, 0.5000001, 0.9]
events = sigma_half_alarm(traj, eps=1e-5)
print("sigma_half_alarm fires at:", events)
print()
print("Same primitive, different boundary_fn -- this is the whole point:")
print("  BoundaryAlarm(is_prime)        -> pi(x), the prime gate")
print("  BoundaryAlarm(|sigma-.5|<eps)  -> Holcus FIRING (wiki/44)")

## Section 2 -- The Prime Instantiation: pi(x)

`gate_alarm` is a fast bisection shortcut. `BoundaryAlarm(is_prime)` reaches
the identical count via the general primitive -- kept here as an equivalence
check, not a separate implementation.

In [ ]:
e = PrimeGateEngine(limit=50000)
print(f"primes up to {e.limit}: {len(e.primes)}")

print("gate_alarm(200) =", e.gate_alarm(200))

alarm = BoundaryAlarm(e.is_prime)
slow_count = alarm.count_at(range(2, 201), 199)
print("BoundaryAlarm(is_prime).count_at equivalent =", slow_count)
print("match:", slow_count == e.gate_alarm(200))

print("alarm_events[:5] =", e.alarm_events()[:5])

## Section 3 -- The Gap Channel: deliberately not part of the alarm

g_n = p_(n+1) - p_n carries everything the alarm leaves out. Kept as a
separate query, only pulled in when a task actually needs spacing.

In [ ]:
fit = e.gap_scaling_fit()
for k, v in fit.items():
    print(f"  {k}: {v}")

## Section 4 -- Two Spirals, Two Addresses

ordinal_spiral addresses by COUNT (n) -- matches the P1 hash convention in
monad.py, and is gap-blind by the same design choice as the alarm itself.
value_spiral addresses by MAGNITUDE (p_n) -- a genuinely different curve.

In [ ]:
import numpy as np
osp = np.array(e.ordinal_spiral())
vsp = np.array(e.value_spiral())

fig, axes = plt.subplots(1, 2, figsize=(11,5.2))
axes[0].plot(osp[:,0], osp[:,1], color="#4af0ff", lw=1.2)
axes[0].plot(0,0,"o",color="#ffb347")
axes[0].set_aspect("equal"); axes[0].set_title("ordinal_spiral T(n) -- address = count")
axes[1].plot(vsp[:,0], vsp[:,1], color="#ff66cc", lw=1.2)
axes[1].plot(0,0,"o",color="#ffb347")
axes[1].set_aspect("equal"); axes[1].set_title("value_spiral T(p_n) -- address = magnitude")
fig.tight_layout()
print("ordinal_spiral[-1] =", tuple(osp[-1]))
print("value_spiral[-1]   =", tuple(vsp[-1]))

## Aside -- Not This Engine's Purpose, Kept on Record

A side investigation happened while building the alarm: does prime-gap
curvature look like a true Euler spiral (clothoid)? curvature_spiral
(heading from kappa_n=ln(p_n), correcting an earlier cumsum(gap_n) attempt
that telescoped trivially back to p_n-p_0) and is_true_euler_spiral answer
no -- a true finding, but a tangent from the alarm, not what this engine
is for.

In [ ]:
cs = np.array(e.curvature_spiral())
result = e.is_true_euler_spiral()
for k, v in result.items():
    print(f"  {k}: {v}")

fig, ax = plt.subplots(figsize=(6,6))
sc = ax.scatter(cs[:,0], cs[:,1], c=np.arange(len(cs)), cmap="cool", s=3)
ax.set_aspect("equal")
ax.set_title("curvature_spiral: kappa_n=ln(p_n) -- single inward spiral, NOT a clothoid")
fig.tight_layout()

## Conclusion

BoundaryAlarm is the engine: a crossing detector, gap/magnitude-blind by
design, reusable across domains (prime gate, Holcus FIRING, or any other
threshold condition). The gap channel, the two spirals, and the Euler-spiral
aside are all things that came up while building it -- not what it is for.